### Example: Walk-Forward Portfolio Optimization

In [1]:
"""
Demonstration of the rolling-window strategy.
"""

import numpy as np
import pandas as pd

import os
os.chdir("..")

# from rl_agent.config import (
#    Config, 
#    EnvironmentConfig, FeatureConfig, 
#    NetworkConfig, TrainingConfig, BacktestConfig
# )
from rl_agent.fin_data import download_fin_data, get_sp500
from rl_agent.forwardbacktest import WalkForwardBacktestEngine
from rl_agent.universe import DynamicUniverse
from rl_agent.utils import (
    compute_portfolio_metrics,
    format_metrics,
    generate_realistic_universe
)

### Obtain data for S&P 500 stocks (replace with your own data loading if needed)

In [2]:
ticker, sp500 = get_sp500()
# _, _, prices = download_fin_data(start_date = "2014-01-01", end_date="2015-12-31", ticker=ticker, sp500=sp500)
# Store returns:
prices = pd.read_csv("C:\\Users\\Lukas\\Downloads\\prices.csv", index_col=0, parse_dates=True)

date       str
tickers    str
dtype: object
1194


In [ ]:
# Replace penny stocks with prices lower 1 with NaN in the prices DataFrame
prices = prices.mask(prices < 1)

# Get returns
returns = prices.pct_change()
returns_win = returns.apply(lambda x: x.clip(lower=x.quantile(0.01), upper=x.quantile(0.99)), axis=1)
# returns_win = returns_win.clip(upper=3, lower=-0.5)
returns_mask = returns != returns_win 
# prices = prices.apply(lambda x: x.clip(lower=x.quantile(0.01), upper=x.quantile(0.99)), axis=1)

# Get adjusted prices
# adjusted_prices = prices.shift(1) * (1 + returns_win)
adjusted_prices = prices[prices.shift(1).isnull()] # Keep price only if first observation 
cum_growth = (1 + returns_win).cumprod()


# Replace outliers
clean_prices = prices.copy()
clean_prices[returns_mask] = adjusted_prices[returns_mask]

In [ ]:
first_prices = prices[prices.shift(1).isnull()] 
adjusted_prices = []

for col in first_prices.columns:
    group_id = first_prices[col].notna().cumsum()
    adj_rets = returns_win[col].mask(first_prices[col].notna(), 0)

    cum_growth = (1 + adj_rets).groupby(group_id).cumprod()
    adj_prices = first_prices[col].groupby(group_id).transform('first')

    adjusted_prices.append((adj_prices * cum_growth).rename(col))

adjusted_prices = pd.concat(adjusted_prices, axis=1)

In [32]:
# Get adjusted prices
# adjusted_prices = prices.shift(1) * (1 + returns_win)
adjusted_prices = prices[prices.shift(1).isnull()] # Keep price only if first observation 
cum_growth = (1 + returns_win.fillna(0)).cumprod()
adjusted_prices = adjusted_prices * cum_growth 

In [65]:
first_prices_MEE = first_prices["MEE"]
ret_MEE = returns["MEE"]
ret_w_MEE = returns_win["MEE"]
prices_MEE = prices["MEE"]
clean_prices_MEE = clean_prices["MEE"]
returns_mask_MEE = returns_mask["MEE"]
adjusted_prices_MEE = adjusted_prices["MEE"]

In [3]:
def _get_clean_prices(prices: pd.DataFrame) -> pd.DataFrame:
    # Replace penny stocks with prices lower 1 with NaN in the prices DataFrame
    prices = prices.mask(prices < 1)

    # Get raw and winsorized returns
    returns = prices.pct_change()
    returns_win = returns.apply(lambda x: x.clip(lower=x.quantile(0.01), upper=x.quantile(0.99)), axis=1)
    # returns_win = returns_win.clip(upper=3, lower=-0.5)
    returns_mask = returns != returns_win
    # prices = prices.apply(lambda x: x.clip(lower=x.quantile(0.01), upper=x.quantile(0.99)), axis=1)

    # Get adjusted prices
    adjusted_prices = []
    first_prices = prices[prices.shift(1).isnull()] # Keep price only if first observation 

    for col in first_prices.columns:
        group_id = first_prices[col].notna().cumsum()
        adj_rets = returns_win[col].mask(first_prices[col].notna(), 0)

        # Calculate cumulative growth for each group of consecutive non-NaN values
        cum_growth = (1 + adj_rets).groupby(group_id).cumprod() 
        adj_prices = first_prices[col].groupby(group_id).transform('first')

        adjusted_prices.append((adj_prices * cum_growth).rename(col))

    adjusted_prices = pd.concat(adjusted_prices, axis=1)

    # Replace outliers
    clean_prices = prices.copy()
    clean_prices[returns_mask] = adjusted_prices[returns_mask]
    
    return clean_prices


In [4]:
clean_prices = _get_clean_prices(prices)

In [5]:
clean_prices_MEE = clean_prices["MEE"]

In [ ]:
returns = prices.pct_change()
weights = pd.read_csv("weights_rl.csv", index_col=0, parse_dates=True)
weights_daily = weights.reindex(returns.index, method='ffill')

In [4]:
retd_march_2014 = returns[(returns.index.year == 2010) & (returns.index.month == 2)]
retd_MEE = returns["MEE"]
prices_MEE = prices["MEE"]

In [12]:
retd_flat_ranked = returns.to_numpy().ravel()
retd_flat_ranked = np.sort(retd_flat_ranked[~np.isnan(retd_flat_ranked)])[::-1]

In [59]:
retd_winsorized = returns.apply(lambda x: x.clip(lower=x.quantile(0.01), upper=x.quantile(0.99)), axis=1)

In [60]:
daily_port_ret = (weights_daily * retd_winsorized).sum(axis=1)
monthly_port_returns = daily_port_ret.groupby(pd.Grouper(freq='ME')).apply(lambda x: (1 + x).prod() - 1)

In [64]:
daily_port_ret_ew = retd_winsorized.mean(axis=1)
monthly_port_returns_ew = daily_port_ret_ew.groupby(pd.Grouper(freq='ME')).apply(lambda x: (1 + x).prod() - 1)

In [66]:
cumulative_returns = (1 + monthly_port_returns.loc['1999-02':]).cumprod() - 1 
cumulative_returns_ew = (1 + monthly_port_returns_ew.loc['1999-02':]).cumprod() - 1

In [3]:
# Initialize backtest engine
engine = WalkForwardBacktestEngine(prices)

In [ ]:
periods = engine._get_periods()
train_start_date, train_end_date, oos_start_date, oos_end_date = periods[0]

In [6]:
active_mask = np.zeros(prices.shape[1], dtype=bool)
active_mask[:prices.shape[1]] = ~prices.isna().loc[oos_start_date].values

In [7]:
# Extract training data (handle NaN for missing data)
window_prices = prices.loc[train_start_date:train_end_date].copy()

In [8]:
from rl_agent.features import FeatureConstructor
from rl_agent.config import FeatureConfig

In [9]:
feature_engine = FeatureConstructor(window_prices, FeatureConfig())

In [11]:
def _build_rebalance_schedule2() -> list[int]:
    dates = window_prices.index
    start_idx = feature_engine.valid_start_idx
    
    # Slice den Index ab dem validen Startpunkt
    relevant_dates = dates[start_idx:]
    
    # Finde Indizes, bei denen der Monat des aktuellen Tages != Monat des nächsten Tages ist
    change_mask = relevant_dates[:-1].month != relevant_dates[1:].month

    # Indizes der Wechsel finden (relativ zum start_idx)
    rebalance_indices = np.where(change_mask)[0] + start_idx
    
    # Letzten Index und Start-Index hinzufügen
    last_idx = len(dates) - 1
    result = np.unique(np.concatenate([[start_idx], rebalance_indices, [last_idx]]))
    
    return result.astype(int).tolist()

In [10]:
from rl_agent.environment import PortfolioEnv
from rl_agent.config import Config, EnvironmentConfig, FeatureConfig, NetworkConfig, TrainingConfig, BacktestConfig

In [11]:
train_env = PortfolioEnv(window_prices, target_returns=EnvironmentConfig().target_returns, 
                         env_config=EnvironmentConfig(), feature_config=FeatureConfig())

In [12]:
lookback = 252 + 100
feature_start = max(prices.index[0], oos_start_date - pd.Timedelta(days=lookback))
feature_prices = prices.loc[feature_start:oos_end_date].copy()

feat_engine = FeatureConstructor(feature_prices, FeatureConfig())

In [13]:
relative_oos_start_idx = feature_prices.index.get_indexer([oos_start_date], method='ffill')[0]
stock_feats = feat_engine.get_stock_features(relative_oos_start_idx)

In [14]:
n_stocks = feature_prices.shape[1]
current_weights = np.zeros(n_stocks, dtype=np.float32)
n_active = active_mask[:n_stocks].sum()
if n_active > 0:
    current_weights[active_mask[:n_stocks]] = 1.0 / n_active

stock_feats = np.hstack([stock_feats, current_weights.reshape(-1, 1)])

# market_feats = feat_engine.get_market_features(relative_oos_start)
market_feats = feat_engine.get_market_features(relative_oos_start_idx)

In [15]:
agent, train_info = engine._train_window(window_prices, active_mask, 0)

In [16]:
# Get deterministic action
action, _, _ = agent.select_action(
    stock_feats, market_feats, deterministic=True
)

In [17]:
# Apply universe mask
action[~active_mask] = -1e6

exp_a = np.exp(action - np.max(action[active_mask]))
exp_a[~active_mask] = 0.0
weights = exp_a / np.sum(exp_a) if np.sum(exp_a) > 1e-10 else np.zeros(n)

# Apply position limits
max_pos = 0.1
weights = np.clip(weights, 0, max_pos)
if weights.sum() > 1e-10:
    weights /= weights.sum()

In [18]:
sw = sum(weights)

In [19]:
oos_returns = feature_prices.loc[oos_start_date:oos_end_date].pct_change()
daily_port_ret = (oos_returns.fillna(0).values @ weights[:n_stocks]).astype(np.float64)

In [20]:
# Apply transaction costs (assume rebalancing from previous weights)
tc_rate = 15 / 10_000
turnover = np.sum(np.abs(weights))  # From cash
tc = turnover * tc_rate

total_raw_return = np.prod(1 + daily_port_ret) - 1

# Subtract TC from first day
if len(daily_port_ret) > 0:
    daily_port_ret[0] -= tc

total_return = np.prod(1 + daily_port_ret) - 1

In [4]:
# Run walk-forward optimization
results = engine.run()

Reinforcement Learning Portfolio Optimization
Dynamic Universe Summary:
  Total unique stocks:        818
  Max concurrent stocks:      500
  Min concurrent stocks:      188
  Mean concurrent stocks:     338
  Entrants:                   474
  Delistings:                 162
  Date range:                 1996-01-02 to 2026-01-30
  Data size:                  818
  Train window:     5.0 years
  Warm-start:       True
  Episodes/window:  200

--- Window 1/306 ---
  Train: 1996-01-02 00:00:00 → 1999-01-29 00:00:00
  OOS:   1999-02-01 00:00:00 → 1999-02-26 00:00:00
  Active stocks: 225
OOS start index: 241 OOS start date: 1999-02-01 00:00:00
OOS return: -0.0105 | OOS raw return: -0.0090 | Train time: 106.9s | Episodes: 114

--- Window 2/306 ---
  Train: 1996-01-02 00:00:00 → 1999-02-26 00:00:00
  OOS:   1999-03-01 00:00:00 → 1999-03-31 00:00:00
  Active stocks: 225
OOS start index: 241 OOS start date: 1999-03-01 00:00:00
OOS return: +0.0167 | OOS raw return: +0.0170 | Train time: 59.5s | E

AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
returns = prices.pct_change().fillna(0)
period_returns = []
active_mask = pd.DataFrame()  
    # for r in results["window_results"]:
    #    period_ret = returns.loc[r.oos_start:r.oos_end]
    #    active = r.active_mask[: prices.shape[1]]
    #    ew_daily = period_ret.iloc[:, active].mean(axis=1)
    #    bm_returns.append(np.prod(1 + ew_daily.values) - 1)
    # for r in results["window_results"]:
    #    start = prices.index.get_indexer([r.oos_start], method="ffill")[0]
    #    end = prices.index.get_indexer([r.oos_end], method="ffill")[0]
    #    period_ret = prices.iloc[start:end + 1].pct_change().fillna(0)
    #    active = r.active_mask[: prices.shape[1]]
    #    ew_daily = period_ret.iloc[:, active].mean(axis=1).values
    #    bm_returns.append(np.prod(1 + ew_daily) - 1)
for r in results["window_results"]:
    active_mask = pd.concat([active_mask, pd.DataFrame([r.active_mask])])
    period_df = returns.loc[r.oos_start:r.oos_end]
    period_returns.append((1 + period_df).prod() - 1)

masked_returns = pd.DataFrame(period_returns).where(active_mask.values, np.nan)
bm_values = np.cumprod(np.concatenate([[1.0], 1 + np.array(masked_returns)]))

bm_metrics = compute_portfolio_metrics(bm_values)
print(format_metrics(bm_metrics))